# 05 - Executive Risk Dashboard View

## Purpose

Create one executive-facing view that combines the advanced Night Shift signals.

This notebook builds a simplified consumption layer for executive-level risk reporting.

## Expected output

`vattenfall_dev.analytics.vw_executive_energy_risk_dashboard`

## Main signals

- market pressure
- weather-grid risk
- operational stress
- executive risk status

## Main idea

Executives should consume clean risk signals, not raw technical complexity.

In [0]:
import sys
import importlib

repo_src_path = "/Workspace/Users/dirella@startsteps.org/vattenfall-week9-capstone-anna/src"

if repo_src_path not in sys.path:
    sys.path.append(repo_src_path)

import risk.risk_classification as risk_classification
importlib.reload(risk_classification)

In [0]:
catalog = "vattenfall_dev"
schema = "analytics"

weather_grid_table = f"{catalog}.{schema}.gold_weather_grid_risk_correlation"
market_ops_table = f"{catalog}.{schema}.gold_market_operations_stress"
asset_intelligence_table = f"{catalog}.{schema}.gold_asset_incident_intelligence"

target_view = f"{catalog}.{schema}.vw_executive_energy_risk_dashboard"

print("Weather-grid source:", weather_grid_table)
print("Market-operations source:", market_ops_table)
print("Asset intelligence source:", asset_intelligence_table)
print("Target executive view:", target_view)

In [0]:
weather_grid_df = spark.table(weather_grid_table)
market_ops_df = spark.table(market_ops_table)
asset_df = spark.table(asset_intelligence_table)

print("Weather-grid rows:", weather_grid_df.count())
print("Market-operations rows:", market_ops_df.count())
print("Asset intelligence rows:", asset_df.count())

display(weather_grid_df)
display(market_ops_df)
display(asset_df)

In [0]:
from pyspark.sql import functions as F

asset_region_risk_df = (
    asset_df
    .groupBy("region")
    .agg(
        F.count("*").alias("asset_count"),
        F.sum(
            F.when(F.col("asset_risk_category").isin("HIGH", "CRITICAL"), 1).otherwise(0)
        ).alias("high_risk_asset_count"),
        F.max("total_incident_count").alias("max_asset_incident_count"),
    )
)

display(asset_region_risk_df)

In [0]:
market_for_exec_df = (
    market_ops_df
    .select(
        "report_day",
        "region",
        F.col("avg_price_eur_mwh").alias("avg_market_price"),
        "max_price_eur_mwh",
        "total_volume_mwh",
        "market_pressure_flag",
        "operations_stress_flag",
        "combined_stress_label",
        F.col("incident_count").alias("market_ops_incident_count"),
        F.col("elevated_incident_count").alias("market_ops_elevated_incident_count"),
        F.col("total_duration_minutes").alias("market_ops_total_duration_minutes"),
    )
)

weather_for_exec_df = (
    weather_grid_df
    .select(
        "report_day",
        "region",
        "weather_grid_risk_score",
        "weather_risk_category",
        "grid_risk_category",
    )
)

asset_for_exec_df = (
    asset_region_risk_df
    .select(
        "region",
        "asset_count",
        "high_risk_asset_count",
        "max_asset_incident_count",
    )
)

executive_df = (
    market_for_exec_df.alias("m")
    .join(
        weather_for_exec_df.alias("w"),
        on=["report_day", "region"],
        how="left"
    )
    .join(
        asset_for_exec_df.alias("a"),
        on="region",
        how="left"
    )
    .fillna(
        {
            "weather_grid_risk_score": 0,
            "asset_count": 0,
            "high_risk_asset_count": 0,
            "max_asset_incident_count": 0,
        }
    )
    .withColumn(
        "executive_risk_status",
        F.when(F.col("combined_stress_label") == "COMBINED_STRESS", "CRITICAL")
        .when(F.col("weather_grid_risk_score") >= 2, "ATTENTION")
        .when(F.col("high_risk_asset_count") > 0, "ATTENTION")
        .when(F.col("combined_stress_label").isin("MARKET_PRESSURE", "OPERATIONS_PRESSURE"), "WATCH")
        .otherwise("NORMAL")
    )
    .select(
        "report_day",
        "region",
        "avg_market_price",
        "max_price_eur_mwh",
        "total_volume_mwh",
        "market_pressure_flag",
        "operations_stress_flag",
        "combined_stress_label",
        "weather_grid_risk_score",
        "weather_risk_category",
        "grid_risk_category",
        F.col("market_ops_incident_count").alias("incident_count"),
        F.col("market_ops_elevated_incident_count").alias("elevated_incident_count"),
        F.col("market_ops_total_duration_minutes").alias("total_duration_minutes"),
        "asset_count",
        "high_risk_asset_count",
        "executive_risk_status",
    )
)

display(executive_df)

In [0]:
executive_table = f"{catalog}.{schema}.gold_executive_energy_risk_dashboard_base"

(
    executive_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(executive_table)
)

spark.sql(f"""
CREATE OR REPLACE VIEW {target_view} AS
SELECT *
FROM {executive_table}
""")

print("Written executive dashboard base table:", executive_table)
print("Created executive risk dashboard view:", target_view)

In [0]:
result_df = spark.table(target_view)

print("Rows in executive risk dashboard view:", result_df.count())
print("Columns:", result_df.columns)

display(result_df)

print("Executive risk status distribution:")
result_df.groupBy("executive_risk_status").count().show()

In [0]:
duplicate_rows = (
    result_df
    .groupBy("report_day", "region")
    .count()
    .filter("count > 1")
    .count()
)

print("Duplicate report_day-region rows:", duplicate_rows)

if duplicate_rows > 0:
    raise ValueError("Executive risk dashboard validation failed: duplicate report_day-region rows found.")

print("Executive risk dashboard view validation passed.")